# Symmetry Sectors with `AbelianBasis`

This notebook demonstrates how EDKit's `AbelianBasis` handles discrete symmetries for exact diagonalization. We cover:

1. **1D chain** -- sector decomposition with `k`, `p`, `z` keywords
2. **2D square lattice** -- arbitrary symmetries via the `symmetries` keyword
3. **Spin inversion** -- sublattice and global spin-flip symmetries
4. **Entanglement** -- Schmidt decomposition in symmetry-reduced bases
5. **Performance** -- Hilbert space reduction factors for practical systems

In [ ]:
import Pkg

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)

using EDKit
using LinearAlgebra

## 1. One-dimensional chain: sector decomposition

The `basis()` constructor accepts keyword arguments `k` (translation momentum), `p` (reflection parity), `z` (spin-flip), and `N` (particle number). Each additional symmetry splits the Hilbert space into smaller sectors.

The key consistency check: eigenvalues from **all** child sectors must recombine to give the parent-sector spectrum.

In [ ]:
L = 10
bond = spin((1.0, "xx"), (1.0, "yy"), (0.5, "zz"))   # XXZ model

# Full spectrum (no symmetry reduction)
full_vals = eigvals(Hermitian(trans_inv_operator(bond, 2, L))) |> sort

# Split into momentum sectors k = 0, 1, ..., L-1
vals_k = Float64[]
for k in 0:L-1
    B = basis(; L, k)
    append!(vals_k, eigvals(Hermitian(trans_inv_operator(bond, 2, B))))
end
println("k-sector recombination error: ", norm(sort(vals_k) - full_vals))

# Further split k=0 into parity p = +/-1
parent = basis(; L, k=0)
parent_vals = eigvals(Hermitian(trans_inv_operator(bond, 2, parent))) |> sort

vals_kp = Float64[]
for p in (1, -1)
    B = basis(; L, k=0, p)
    append!(vals_kp, eigvals(Hermitian(trans_inv_operator(bond, 2, B))))
end
println("(k,p)-sector recombination error: ", norm(sort(vals_kp) - parent_vals))

# Full decomposition at half-filling: N, k=0, p, z
N = L ÷ 2
parent_Nk = basis(; L, N, k=0)
parent_Nk_vals = eigvals(Hermitian(trans_inv_operator(bond, 2, parent_Nk))) |> sort

vals_full = Float64[]
for p in (1, -1), z in (1, -1)
    B = basis(; L, N, k=0, p, z)
    iszero(size(B, 1)) && continue
    append!(vals_full, eigvals(Hermitian(trans_inv_operator(bond, 2, B))))
end
println("(N,k,p,z)-sector recombination error: ", norm(sort(vals_full) - parent_Nk_vals))

println("\nDimension breakdown at (N=$N, k=0):")
println("  Parent dim = ", size(parent_Nk, 1))
for p in (1, -1), z in (1, -1)
    B = basis(; L, N, k=0, p, z)
    println("  (p=$p, z=$z): dim = ", size(B, 1))
end

## 2. Two-dimensional lattice: the `symmetries` keyword

For lattices beyond 1D, the `k`/`p`/`z` keywords are not enough. The `symmetries` keyword accepts a list of `(permutation_array, quantum_number)` tuples, where each permutation defines how sites map under a symmetry operation.

**Permutation format**: `perm[i] = j` means site `i` maps to site `j` (1-indexed). The group order is auto-computed from the permutation period.

In [ ]:
# 3x3 square lattice with periodic boundary conditions
Lx, Ly = 3, 3
L = Lx * Ly
sites = [(x, y) for y in 0:Ly-1 for x in 0:Lx-1]

# Translation generators (permutation arrays)
T_x = [mod(x + 1, Lx) + Lx * y + 1 for (x, y) in sites]   # shift in x
T_y = [x + Lx * mod(y + 1, Ly) + 1 for (x, y) in sites]   # shift in y

println("Lattice sites (row-major): ", sites)
println("T_x permutation: ", T_x)
println("T_y permutation: ", T_y)

In [ ]:
# Build 2D Heisenberg Hamiltonian (nearest-neighbor bonds)
J = spin((1.0, "xx"), (1.0, "yy"), (1.0, "zz"))

bonds = Tuple{Int,Int}[]
for (x, y) in sites
    i = x + Lx * y + 1
    push!(bonds, (i, mod(x + 1, Lx) + Lx * y + 1))       # x-bond
    push!(bonds, (i, x + Lx * mod(y + 1, Ly) + 1))         # y-bond
end

# Full-space spectrum
H_full = operator([J for _ in bonds], [[b[1], b[2]] for b in bonds], L)
full_vals = eigvals(Hermitian(Array(H_full))) |> sort

# Reconstruct from all (kx, ky) momentum sectors
vals_2d = Float64[]
dims_2d = Dict{Tuple{Int,Int}, Int}()
for kx in 0:Lx-1, ky in 0:Ly-1
    B = basis(; L, symmetries=[(T_x, kx), (T_y, ky)])
    dims_2d[(kx, ky)] = size(B, 1)
    iszero(size(B, 1)) && continue
    H = operator([J for _ in bonds], [[b[1], b[2]] for b in bonds], B)
    append!(vals_2d, eigvals(Hermitian(Array(H))))
end

println("2D spectrum recombination error: ", norm(sort(vals_2d) - full_vals))
println("\nSector dimensions (kx, ky) -> dim:")
for ky in 0:Ly-1
    for kx in 0:Lx-1
        print("  ($kx,$ky):$(dims_2d[(kx,ky)])")
    end
    println()
end
println("Total: ", sum(values(dims_2d)), " (full space: ", 2^L, ")")

### Combining 2D translations with particle conservation

The `N` keyword works alongside `symmetries`, restricting to states with a fixed number of particles. This combines U(1) charge conservation with the lattice symmetries.

In [ ]:
# Half-filling sector with 2D momentum
N = L ÷ 2
B_N = basis(; L, N)   # particle conservation only
ref_vals = eigvals(Hermitian(Array(operator([J for _ in bonds], [[b[1], b[2]] for b in bonds], B_N)))) |> sort

vals_Nk = Float64[]
for kx in 0:Lx-1, ky in 0:Ly-1
    B = basis(; L, N, symmetries=[(T_x, kx), (T_y, ky)])
    iszero(size(B, 1)) && continue
    H = operator([J for _ in bonds], [[b[1], b[2]] for b in bonds], B)
    append!(vals_Nk, eigvals(Hermitian(Array(H))))
end

println("(N, kx, ky) recombination error: ", norm(sort(vals_Nk) - ref_vals))
println("Full N=$N sector dim: ", size(B_N, 1))

# Find ground state sector
E_min = Inf
gs_sector = (0, 0)
for kx in 0:Lx-1, ky in 0:Ly-1
    B = basis(; L, N, symmetries=[(T_x, kx), (T_y, ky)])
    iszero(size(B, 1)) && continue
    H = operator([J for _ in bonds], [[b[1], b[2]] for b in bonds], B)
    e = eigvals(Hermitian(Array(H)))[1]
    if e < E_min
        E_min = e
        gs_sector = (kx, ky)
    end
end
println("Ground state at (kx, ky) = $gs_sector, E = $(round(E_min, digits=6))")

## 3. Spin inversion via the `symmetries` keyword

Spin-flip symmetry (each spin flipped: $|\uparrow\rangle \leftrightarrow |\downarrow\rangle$) can be specified as a third element in the symmetry tuple: `(perm, q, inv_mask)` where `inv_mask` is a `BitVector` indicating which sites get spin-inverted.

For global spin-flip, use the identity permutation with all sites inverted.

In [ ]:
# Global spin-flip via symmetries keyword (equivalent to z=+/-1)
L_flip = 8
bond_flip = spin((1.0, "xx"), (1.0, "yy"), (0.5, "zz"))

# Using the dedicated z keyword
B_z1 = basis(; L=L_flip, N=L_flip÷2, k=0, z=1)
B_z2 = basis(; L=L_flip, N=L_flip÷2, k=0, z=-1)

# Using symmetries keyword with spin inversion
perm_id = collect(1:L_flip)
inv_all = trues(L_flip)

# Translation + spin flip combined
T_1d = [mod1(i + 1, L_flip) for i in 1:L_flip]
B_sym1 = basis(; L=L_flip, N=L_flip÷2, symmetries=[(T_1d, 0), (perm_id, 0, inv_all)])
B_sym2 = basis(; L=L_flip, N=L_flip÷2, symmetries=[(T_1d, 0), (perm_id, 1, inv_all)])

println("Dedicated (k=0, z=+1) dim: ", size(B_z1, 1))
println("Symmetries (k=0, z=+1) dim: ", size(B_sym1, 1))
println("Dedicated (k=0, z=-1) dim: ", size(B_z2, 1))
println("Symmetries (k=0, z=-1) dim: ", size(B_sym2, 1))

# Verify eigenvalues match
E_ded = eigvals(Hermitian(trans_inv_operator(bond_flip, 2, B_z1))) |> sort
E_sym = eigvals(Hermitian(trans_inv_operator(bond_flip, 2, B_sym1))) |> sort
println("\nEigenvalue match (z=+1): ", norm(E_ded - E_sym) < 1e-12 ? "PASS" : "FAIL")

## 4. Entanglement in symmetry-reduced bases

The `ent_S` function computes bipartite entanglement entropy directly from eigenvectors in any symmetry-reduced basis. The Schmidt decomposition correctly handles the orbit expansion.

In [ ]:
# Entanglement entropy: 2D lattice ground state
Lx_e, Ly_e = 3, 2
L_e = Lx_e * Ly_e
sites_e = [(x, y) for y in 0:Ly_e-1 for x in 0:Lx_e-1]
T_xe = [mod(x + 1, Lx_e) + Lx_e * y + 1 for (x, y) in sites_e]
T_ye = [x + Lx_e * mod(y + 1, Ly_e) + 1 for (x, y) in sites_e]

bonds_e = Tuple{Int,Int}[]
for (x, y) in sites_e
    i = x + Lx_e * y + 1
    push!(bonds_e, (i, mod(x + 1, Lx_e) + Lx_e * y + 1))
    push!(bonds_e, (i, x + Lx_e * mod(y + 1, Ly_e) + 1))
end
J_e = spin((1.0, "xx"), (1.0, "yy"), (1.0, "zz"))

# Full-space ground state
B_full_e = TensorBasis(; L=L_e, base=2)
H_full_e = operator([J_e for _ in bonds_e], [[b[1], b[2]] for b in bonds_e], L_e)
E_full_e, V_full_e = eigen(Hermitian(Array(H_full_e)))
gs_full = V_full_e[:, 1]
S_full = ent_S(gs_full, 1:L_e÷2, B_full_e)

# Symmetry-reduced ground state (scan all sectors to find it)
E_min_e = Inf
S_reduced = NaN
for kx in 0:Lx_e-1, ky in 0:Ly_e-1
    B = basis(; L=L_e, symmetries=[(T_xe, kx), (T_ye, ky)])
    iszero(size(B, 1)) && continue
    H = operator([J_e for _ in bonds_e], [[b[1], b[2]] for b in bonds_e], B)
    e, v = eigen(Hermitian(Array(H)))
    if e[1] < E_min_e
        E_min_e = e[1]
        S_reduced = ent_S(v[:, 1], 1:L_e÷2, B)
    end
end

println("Ground state energy:")
println("  Full space:    ", round(E_full_e[1], digits=10))
println("  Reduced basis: ", round(E_min_e, digits=10))
println("\nEntanglement entropy (half-chain bipartition):")
println("  Full space:    ", round(S_full, digits=10))
println("  Reduced basis: ", round(S_reduced, digits=10))
println("  Match: ", abs(S_full - S_reduced) < 1e-10 ? "PASS" : "FAIL")

## 5. Hilbert space reduction

Symmetry reduction is the whole point: the largest sector is much smaller than the full space, enabling exact diagonalization of systems that would otherwise be intractable.

In [ ]:
# Dimension reduction table for 1D chains
println("=== 1D Chain Reduction ===")
println(rpad("L", 5), rpad("Full", 8), rpad("N=L/2", 8), rpad("+k=0", 8), rpad("+p=1", 8), rpad("+z=1", 8))
for L in [8, 10, 12, 14, 16, 18, 20]
    d_full = 2^L
    N = L ÷ 2
    d_N = size(basis(; L, N), 1)
    d_Nk = size(basis(; L, N, k=0), 1)
    d_Nkp = size(basis(; L, N, k=0, p=1), 1)
    d_Nkpz = size(basis(; L, N, k=0, p=1, z=1), 1)
    println(rpad(L, 5), rpad(d_full, 8), rpad(d_N, 8), rpad(d_Nk, 8), rpad(d_Nkp, 8), rpad(d_Nkpz, 8))
end

println("\n=== 2D Square Lattice Reduction (kx=0, ky=0, N=L/2) ===")
println(rpad("Lattice", 10), rpad("L", 5), rpad("Full", 10), rpad("N=L/2", 10), rpad("+(kx,ky)", 10))
for (Lx, Ly) in [(2,3), (3,3), (4,3), (3,4), (4,4)]
    L = Lx * Ly
    L > 20 && continue
    N = L ÷ 2
    d_full = 2^L
    d_N = binomial(L, N)
    sites = [(x, y) for y in 0:Ly-1 for x in 0:Lx-1]
    Tx = [mod(x+1, Lx) + Lx*y + 1 for (x,y) in sites]
    Ty = [x + Lx*mod(y+1, Ly) + 1 for (x,y) in sites]
    B2d = basis(; L, N, symmetries=[(Tx, 0), (Ty, 0)])
    d_k = size(B2d, 1)
    println(rpad("$(Lx)x$(Ly)", 10), rpad(L, 5), rpad(d_full, 10), rpad(d_N, 10), rpad(d_k, 10))
end

## Summary

The `basis()` constructor provides two ways to specify symmetries:

- **1D keywords** (`k`, `p`, `z`): convenient for periodic chains
- **`symmetries` keyword**: arbitrary permutation-based generators for any lattice geometry

Both approaches produce an `AbelianBasis` that works seamlessly with `operator()`, `ent_S()`, and all other EDKit functions. The key checks:

1. **Dimension**: child sectors must sum to the parent sector dimension
2. **Spectrum**: merged child-sector eigenvalues must reproduce the parent spectrum
3. **Entanglement**: `ent_S` gives consistent results in any symmetry-reduced basis